In [1]:
from google.colab import files
uploaded=files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv


In [4]:
import pandas as pd
import numpy as np

In [5]:
df=pd.read_csv("train.csv")
test=pd.read_csv('test.csv')

In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [9]:
print(df.shape)

(594194, 21)


In [18]:
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True))

Churn
No     460377
Yes    133817
Name: count, dtype: int64
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64


In [13]:
df.isnull().sum()

,0
id,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [19]:
cat_cols=df.select_dtypes(include='object').columns
for col in cat_cols:
  print("\n",col)
  print(df[col].value_counts())


 gender
gender
Female    298738
Male      295456
Name: count, dtype: int64

 Partner
Partner
Yes    309554
No     284640
Name: count, dtype: int64

 Dependents
Dependents
No     414362
Yes    179832
Name: count, dtype: int64

 PhoneService
PhoneService
Yes    557893
No      36301
Name: count, dtype: int64

 MultipleLines
MultipleLines
No                  283384
Yes                 274509
No phone service     36301
Name: count, dtype: int64

 InternetService
InternetService
Fiber optic    272386
DSL            181081
No             140727
Name: count, dtype: int64

 OnlineSecurity
OnlineSecurity
No                     289474
Yes                    163993
No internet service    140727
Name: count, dtype: int64

 OnlineBackup
OnlineBackup
No                     250083
Yes                    203384
No internet service    140727
Name: count, dtype: int64

 DeviceProtection
DeviceProtection
No                     247377
Yes                    206090
No internet service    140727
Name: count

In [16]:
df.describe()

,id,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,594194.000000,594194.000000,594194.000000,594194.000000,594194.000000
mean,297096.500000,0.114102,36.577258,65.866223,2494.377057
std,171529.177262,0.317936,25.061922,31.067444,2353.916710
min,0.000000,0.000000,1.000000,18.250000,18.800000
25%,148548.250000,0.000000,12.000000,29.900000,639.650000
50%,297096.500000,0.000000,35.000000,74.100000,1433.650000
75%,445644.750000,0.000000,62.000000,90.800000,4263.800000
max,594193.000000,1.000000,72.000000,118.750000,8684.800000


In [21]:
pd.crosstab(df['Contract'],df['Churn'],normalize='index')

Churn,No,Yes
Contract,,
Month-to-month,0.579457,0.420543
One year,0.942372,0.057628
Two year,0.990018,0.009982


In [22]:
pd.crosstab(df['InternetService'], df['Churn'], normalize='index')

Churn,No,Yes
InternetService,,
DSL,0.896936,0.103064
Fiber optic,0.584634,0.415366
No,0.985689,0.014311


In [23]:
pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index')

Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),0.922907,0.077093
Credit card (automatic),0.930668,0.069332
Electronic check,0.510948,0.489052
Mailed check,0.920303,0.079697


In [26]:
df['Churn'] = df['Churn'].str.strip()
y = df['Churn'].map({'No': 0, 'Yes': 1})

X = df.drop('Churn', axis=1)

test_ids = test['id']

X = X.drop('id', axis=1)
test = test.drop('id', axis=1)

In [28]:
# 1. tenure grouping
X['tenure_group'] = pd.cut(
    X['tenure'],
    bins=[0,12,24,48,72],
    labels=['0-12','12-24','24-48','48-72'],
    include_lowest=True
)

test['tenure_group'] = pd.cut(
    test['tenure'],
    bins=[0,12,24,48,72],
    labels=['0-12','12-24','24-48','48-72'],
    include_lowest=True
)

# 2. ratios
X['charge_ratio'] = X['MonthlyCharges'] / (X['TotalCharges'] + 1)
test['charge_ratio'] = test['MonthlyCharges'] / (test['TotalCharges'] + 1)

X['avg_charge'] = X['TotalCharges'] / (X['tenure'] + 1)
test['avg_charge'] = test['TotalCharges'] / (test['tenure'] + 1)

# 3. high-risk features
X['is_fiber'] = (X['InternetService'] == 'Fiber optic').astype(int)
test['is_fiber'] = (test['InternetService'] == 'Fiber optic').astype(int)

X['is_electronic'] = (X['PaymentMethod'] == 'Electronic check').astype(int)
test['is_electronic'] = (test['PaymentMethod'] == 'Electronic check').astype(int)

In [29]:
print(X.head())

   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0    Male              0     Yes        Yes      29          Yes   
1    Male              0     Yes        Yes      58          Yes   
2    Male              0     Yes         No      58          Yes   
3  Female              0      No         No       1          Yes   
4  Female              0      No         No       1          Yes   

  MultipleLines InternetService OnlineSecurity OnlineBackup  ...  \
0            No             DSL            Yes           No  ...   
1            No             DSL            Yes          Yes  ...   
2           Yes     Fiber optic             No          Yes  ...   
3            No     Fiber optic             No           No  ...   
4            No     Fiber optic             No           No  ...   

         Contract PaperlessBilling            PaymentMethod MonthlyCharges  \
0        One year              Yes             Mailed check          60.10   
1        Two year         

In [30]:
X = pd.get_dummies(X, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

X, test = X.align(test, join='left', axis=1, fill_value=0)

In [31]:
print(X.shape)
print(X.select_dtypes(include='object').columns)

(594194, 37)
Index([], dtype='object')


In [32]:
print(X.isnull().sum().sum())
print(test.isnull().sum().sum())

0
0


In [33]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [34]:
print(X_train.shape, X_val.shape)

(475355, 37) (118839, 37)


In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
ratio = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(scale_pos_weight=ratio, eval_metric='logloss', random_state=42)
}
def evaluate_model(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)
results = {}

for name, model in models.items():
    score = evaluate_model(model, X_train, y_train, X_val, y_val)
    results[name] = score
    print(f"{name}: ROC-AUC = {score:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic: ROC-AUC = 0.9111
RandomForest: ROC-AUC = 0.9011
XGBoost: ROC-AUC = 0.9158


In [36]:
final_model = XGBClassifier(
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    eval_metric='logloss',
    random_state=42
)

final_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [37]:
test_preds = final_model.predict_proba(test)[:, 1]

In [38]:
submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_preds
})

submission.to_csv("submission.csv", index=False)

In [40]:
print(submission.shape)
submission.head()

(254655, 2)


,id,Churn
0,594194,0.239360
1,594195,0.000434
2,594196,0.298194
3,594197,0.007139
4,594198,0.834795


In [41]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>